In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as sklearn

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputData = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIG = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
# Convert back to timedelta
timeSeriesData_BIG['timestamp'] = pd.to_timedelta(timeSeriesData_BIG['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIG ["datetime_timestamp"]= timeSeriesData_BIG['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


In [ ]:
columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment', ]
columns_timedelta = ['time taken total',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']

userInputData.loc[:,columns_datetime] = userInputData.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputData.loc[:,columns_timedelta] = userInputData.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
# Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}

In [ ]:
userInputData

In [ ]:
userInputData["room"].unique()

In [ ]:
userInputData_onlyRoomTogether = userInputData.loc[(userInputData["room"] == 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1') & (userInputData['experimentState']=='InsertingSourcePollutant'),:]
userInputData_onlyRoomTogether = userInputData_onlyRoomTogether.reset_index(names = "keys")
userInputData_onlyRoomTogether

In [ ]:
timeSeriesData_BIG_onlyRoomTogether = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData_onlyRoomTogether["keys"]),:]
timeSeriesData_BIG_onlyRoomTogether

In [ ]:
min_sec = 30
max_sec = 120
total=min_sec + max_sec 

In [ ]:


dict_of_timeseries_onlyRoomTogether_Data = {}

groups_keys = timeSeriesData_BIG_onlyRoomTogether.groupby("keys").groups

for key, indexes_of_the_timeSeriesData_BIG in groups_keys.items(): 
        min_second = (userInputData_onlyRoomTogether.loc[userInputData_onlyRoomTogether["keys"]==key,"timestamp InsertingSource seconds"].iloc[0]) - min_sec
        max_second = (userInputData_onlyRoomTogether.loc[userInputData_onlyRoomTogether["keys"]==key,"timestamp InsertingSource seconds"].iloc[0]) + max_sec
        particular_experiment = timeSeriesData_BIG_onlyRoomTogether.loc[timeSeriesData_BIG_onlyRoomTogether.index.isin(indexes_of_the_timeSeriesData_BIG)]
        mapped_rows = particular_experiment["seconds"].map(lambda x:True if ((x>=min_second) and (x<max_second)) else False) 
        dict_of_timeseries_onlyRoomTogether_Data[key]  =  particular_experiment.loc[mapped_rows,["seconds","sensors","VOC"]] 


In [ ]:
dict_of_timeseries_onlyRoomTogether_Data

In [ ]:
for key,df_data in dict_of_timeseries_onlyRoomTogether_Data.items():
    print(df_data.shape)

In [ ]:
for key,df_data in dict_of_timeseries_onlyRoomTogether_Data.items():
    data = df_data.pivot(columns='sensors',index='seconds',values="VOC")
    data.columns.name = None
    
    data = data.reset_index(drop=True)
    data.index.name = "seconds"
    
    dict_of_timeseries_onlyRoomTogether_Data[key] = data
    

In [ ]:
#dict_of_timeseries_onlyRoomTogether_Data[20]

In [ ]:
for key,df_data in dict_of_timeseries_onlyRoomTogether_Data.items():
    print(df_data.shape)

In [ ]:
current_shape = (total, 2)

dict_of_timeseries_onlyRoomTogether_Data = {
    k: v for k, v in dict_of_timeseries_onlyRoomTogether_Data.items()
    if v.shape == current_shape
}
        

In [ ]:
for key,df_data in dict_of_timeseries_onlyRoomTogether_Data.items():
    print(df_data.shape)

In [ ]:
userInputData_onlyRoomTogether

In [ ]:
dict_of_timeseries_onlyRoomTogether_Data

In [ ]:
x_time_series = []
y_source_clasiffier = []
for key,df_data in dict_of_timeseries_onlyRoomTogether_Data.items():
       mask = userInputData_onlyRoomTogether["keys"]==key
       y_element = userInputData_onlyRoomTogether.loc[mask,["x axis","y axis"]].to_numpy()
       y_source_clasiffier.append(y_element)
       x_element = df_data.to_numpy()
       x_time_series.append(x_element)

In [ ]:
x_time_series

In [ ]:
y_source_clasiffier

In [ ]:
for data in x_time_series:
        # Check if all values are strictly positive
    all_positive = np.all(data > 0)
    print(all_positive)  # True

In [ ]:
for i in range(len(x_time_series)):
    x_time_series[i] = np.abs(x_time_series[i])


In [ ]:
for data in x_time_series:
        # Check if all values are strictly positive
    all_positive = np.all(data > 0)
    print(all_positive)  # True

In [ ]:
X_features = [exp.flatten() for exp in x_time_series]  # shape (n_experiments, 900)

In [ ]:
X_features[1].shape

In [ ]:
y_array = np.vstack(y_source_clasiffier)   # shape (n_experiments, 2)
_, y_labels = np.unique(y_array, axis=0, return_inverse=True)
# now y_labels is 1D, e.g. [0,0,0,0,1,0,1,2,...]

In [ ]:
y_labels

In [ ]:
#y_labels = [0 if label != 3 else 3 for label in y_labels]

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
# Train classifier
clf = SVC(kernel='rbf')
clf.fit(X_train, y_train)
score =clf.score(X_test, y_test)
print("Test accuracy:", score)

print("Test accuracy:", clf.score(X_test, y_test))

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def test_svc(room_name):
    min_sec = 30
    max_sec = 120
    score_results = {}
    
    userInputData_onlyRoomTogether = userInputData.loc[(userInputData["room"] == room_name) & (userInputData['experimentState']=='InsertingSourcePollutant'),:]
    userInputData_onlyRoomTogether = userInputData_onlyRoomTogether.reset_index(names = "keys")
    timeSeriesData_BIG_onlyRoomTogether = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData_onlyRoomTogether["keys"]),:]
    positions = 0
    y_labels = 0
    for max_sec in range(30,300,30):
        
        total=min_sec + max_sec 
        print(f"total seconds:{total}")
        dict_of_timeseries_onlyRoomTogether_Data = {}
        groups_keys = timeSeriesData_BIG_onlyRoomTogether.groupby("keys").groups
        for key, indexes_of_the_timeSeriesData_BIG in groups_keys.items(): 
                min_second = (userInputData_onlyRoomTogether.loc[userInputData_onlyRoomTogether["keys"]==key,"timestamp InsertingSource seconds"].iloc[0]) - min_sec
                max_second = (userInputData_onlyRoomTogether.loc[userInputData_onlyRoomTogether["keys"]==key,"timestamp InsertingSource seconds"].iloc[0]) + max_sec
                particular_experiment = timeSeriesData_BIG_onlyRoomTogether.loc[timeSeriesData_BIG_onlyRoomTogether.index.isin(indexes_of_the_timeSeriesData_BIG)]
                mapped_rows = particular_experiment["seconds"].map(lambda x:True if ((x>=min_second) and (x<max_second)) else False) 
                dict_of_timeseries_onlyRoomTogether_Data[key]  =  particular_experiment.loc[mapped_rows,["seconds","sensors","VOC"]] 
             #   print(dict_of_timeseries_onlyRoomTogether_Data[key].shape)
            
        for key,df_data in dict_of_timeseries_onlyRoomTogether_Data.items():
            data = df_data.pivot(columns='sensors',index='seconds',values="VOC")
            data.columns.name = None
            
            data = data.reset_index(drop=True)
            data.index.name = "seconds"
            dict_of_timeseries_onlyRoomTogether_Data[key] = data
      #  print(f"current_shape{current_shape}")
        
    
        
        current_shape = (total, 2)
    
        dict_of_timeseries_onlyRoomTogether_Data = {
            k: v for k, v in dict_of_timeseries_onlyRoomTogether_Data.items()
            if v.shape == current_shape
        }
            
        x_time_series = []
        y_source_clasiffier = []
    #    print(dict_of_timeseries_onlyRoomTogether_Data)
        for key,df_data in dict_of_timeseries_onlyRoomTogether_Data.items():
               mask = userInputData_onlyRoomTogether["keys"]==key
               y_element = userInputData_onlyRoomTogether.loc[mask,["x axis","y axis"]].to_numpy()
    
               y_source_clasiffier.append(y_element)
               x_element = df_data.to_numpy()
               x_time_series.append(x_element)   
        for i in range(len(x_time_series)):
            x_time_series[i] = np.abs(x_time_series[i])
        
        
      #  print(y_source_clasiffier)
        X_features = [exp.flatten() for exp in x_time_series]
        y_array = np.vstack(y_source_clasiffier)   # shape (n_experiments, 2)
        positions, y_labels = np.unique(y_array, axis=0, return_inverse=True)
        print(f"positions:{positions}")
        print(f"y_labels{y_labels}")
            # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X_features, y_labels, test_size=0.2, random_state=42, stratify=y_labels
        )
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        # Train classifier
        clf = SVC(kernel='rbf')
        clf.fit(X_train, y_train)
        score =clf.score(X_test, y_test)
        
        print("Test accuracy:", score)
        y_pred = clf.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(cmap='Blues')  # nicer colors
    return positions,y_labels 

In [ ]:
userInputData["room"].unique()

In [ ]:
room_name = 'Κρεβατοκάμαρα id:1 Π0.6 Α1.2 id:2 Μ0.8 Α1.1'
positions, y_labels= test_svc(room_name)

In [ ]:
userInputData_onlyRoomTogether

In [ ]:
positions,y_labels

In [ ]:
# Extra points
position_of_sensors = userInputData.loc[userInputData["room"] == room_name].iloc[0]

extra_positions = np.array([
    [position_of_sensors["position of Id=1:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=1:BME680:breathVocEquivalent-y axis"]],
    [position_of_sensors["position of Id=2:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=2:BME680:breathVocEquivalent-y axis"]]
])
extra_ids = ["id1", "id2"]

# Example max values (you can replace with user input)
room_length = 2.0
room_width = 1.85 

# Create scatterplot
#sns.scatterplot(x=positions[:,0], y=positions[:,1])


# Add extra points
sns.scatterplot(x=extra_positions[:,0], y=extra_positions[:,1], color='red', s=100)

# Annotate extra points with their IDs
for i, (x, y) in enumerate(extra_positions):
    plt.text(x, y, extra_ids[i], fontsize=12, ha='right', va='bottom', color='red')


# Set axis limits
plt.xlim(-room_width, room_width)
plt.ylim(-room_length, room_length)

# Add grid
plt.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)

plt.show()

In [ ]:
positions,y_labels